In [1]:
import os
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
import pandas as pd
import time
import csv
import gzip

# Carga .env desde la raíz del proyecto (útil en local; en Docker las vars ya están inyectadas)
load_dotenv(find_dotenv())

URI = os.getenv("NEO4J_URI")
AUTH = (os.getenv("NEO4J_USER"), os.getenv("NEO4J_PASSWORD"))


In [2]:

try:
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        driver.verify_connectivity()
        print("¡Conexión exitosa a tu Neo4j local en Docker! 🚀")
except Exception as e:
    print(f"Error al conectar: {e}")

¡Conexión exitosa a tu Neo4j local en Docker! 🚀


In [3]:
df = pd.read_csv("../data/raw/ml_dataset.csv")

df

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.00,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.00,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.00,0.00,0,0
...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.00,C776919290,0.00,339682.13,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.00,C1881841831,0.00,0.00,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.00,C1365125890,68488.84,6379898.11,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.00,C2080388513,0.00,0.00,1,0


In [5]:

def ingestar_raw_data_optimizada(df, batch_size=50000):
    """
    Ingesta más de 6M de filas optimizando el uso de RAM en Python 
    y reduciendo al mínimo los bloqueos (locks) en Neo4j.
    """
    
    # 1. Queries optimizadas por fases
    query_nodos_origen = """
    UNWIND $rows AS row
    MERGE (:Account {id: row.nameOrig})
    """
    
    query_nodos_destino = """
    UNWIND $rows AS row
    MERGE (:Account {id: row.nameDest})
    """
    
    query_relaciones = """
    UNWIND $rows AS row
    MATCH (orig:Account {id: row.nameOrig})
    MATCH (dest:Account {id: row.nameDest})
    CREATE (orig)-[:TRANSACTION {
        amount: toFloat(row.amount),
        type: row.type,
        step: toInteger(row.step),
        oldbalanceOrg: toFloat(row.oldbalanceOrg),
        newbalanceOrig: toFloat(row.newbalanceOrig),
        oldbalanceDest: toFloat(row.oldbalanceDest),
        newbalanceDest: toFloat(row.newbalanceDest),
        isFraud: toInteger(row.isFraud)
    }]->(dest)
    """

    total_filas = len(df)
    print(f"Preparando ingesta masiva de {total_filas} registros...")
    
    start_time = time.time()
    
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:
            
            # FASE 0: Asegurar restricciones de unicidad e índices (Vital para la velocidad)
            print("[Fase 0] Creando restricciones de unicidad...")
            session.run("CREATE CONSTRAINT ACCOUNT_ID_UNIQUE IF NOT EXISTS FOR (a:Account) REQUIRE a.id IS UNIQUE")
            
            # FASE 1: Crear los Nodos (Origen y Destino por separado para evitar cuellos de botella)
            print("\n[Fase 1] Cargando nodos Account únicos a la base de datos...")
            
            # Extraemos IDs únicos para no enviar duplicados a Neo4j en la fase de nodos
            origenes_unicos = df[['nameOrig']].drop_duplicates()
            destinos_unicos = df[['nameDest']].drop_duplicates()
            
            # Cargar orígenes
            for i in range(0, len(origenes_unicos), batch_size):
                chunk = origenes_unicos.iloc[i:i + batch_size]
                batch = chunk.to_dict('records')
                session.execute_write(lambda tx: tx.run(query_nodos_origen, rows=batch))
            
            # Cargar destinos
            for i in range(0, len(destinos_unicos), batch_size):
                chunk = destinos_unicos.iloc[i:i + batch_size]
                batch = chunk.to_dict('records')
                session.execute_write(lambda tx: tx.run(query_nodos_destino, rows=batch))
                
            print(f"-> Nodos asegurados en el grafo. Tiempo: {round(time.time() - start_time, 2)} s")
            
            # FASE 2: Crear las Relaciones (La parte masiva de 6.3M)
            print("\n[Fase 2] Creando las 6M+ relaciones TRANSACTION (Carga por lotes)...")
            rel_start_time = time.time()
            
            for i in range(0, total_filas, batch_size):
                # Extraemos el bloque directamente de Pandas
                chunk = df.iloc[i:i + batch_size]
                
                # Convertimos a diccionario SOLO las columnas que van a la relación para liberar RAM
                batch_records = chunk[[
                    'nameOrig', 'nameDest', 'amount', 'type', 'step', 
                    'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud'
                ]].to_dict('records')
                
                # Inyección explícita de escritura en Neo4j
                session.execute_write(lambda tx: tx.run(query_relaciones, rows=batch_records))
                
                # Reporte de progreso cada 500,000 registros
                if i % 500000 == 0 and i > 0:
                    elapsed = time.time() - rel_start_time
                    print(f"   Progreso: {i}/{total_filas} relaciones inyectadas... [{round(elapsed, 2)} s]")
            
            end_time = time.time()
            print(f"\n--- INGESTA COMPLETA ---")
            print(f"Tiempo total del proceso: {round((end_time - start_time)/60, 2)} minutos.")

ingestar_raw_data_optimizada(df)

Preparando ingesta masiva de 6362620 registros...
[Fase 0] Creando restricciones de unicidad...

[Fase 1] Cargando nodos Account únicos a la base de datos...
-> Nodos asegurados en el grafo. Tiempo: 190.18 s

[Fase 2] Creando las 6M+ relaciones TRANSACTION (Carga por lotes)...
   Progreso: 500000/6362620 relaciones inyectadas... [30.58 s]
   Progreso: 1000000/6362620 relaciones inyectadas... [57.99 s]
   Progreso: 1500000/6362620 relaciones inyectadas... [85.66 s]
   Progreso: 2000000/6362620 relaciones inyectadas... [113.07 s]
   Progreso: 2500000/6362620 relaciones inyectadas... [141.25 s]
   Progreso: 3000000/6362620 relaciones inyectadas... [169.76 s]
   Progreso: 3500000/6362620 relaciones inyectadas... [198.09 s]
   Progreso: 4000000/6362620 relaciones inyectadas... [227.57 s]
   Progreso: 4500000/6362620 relaciones inyectadas... [256.69 s]
   Progreso: 5000000/6362620 relaciones inyectadas... [285.78 s]
   Progreso: 5500000/6362620 relaciones inyectadas... [315.2 s]
   Progreso:

In [16]:
def verificar_datos_neo4j():
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:
            
            # 1️⃣ Contar el número de nodos Account
            nodos_count = session.run("MATCH (a:Account) RETURN count(a) AS total").single()["total"]
            print(f"Número total de nodos Account: {nodos_count}")
            
            # 2️⃣ Contar el número de relaciones TRANSACTION
            relaciones_count = session.run("MATCH (:Account)-[t:TRANSACTION]->(:Account) RETURN count(t) AS total").single()["total"]
            print(f"Número total de relaciones TRANSACTION: {relaciones_count}")
            
            # 3️⃣ Verificar algunas relaciones aleatorias
            ejemplo = session.run("""
                MATCH (a:Account)-[t:TRANSACTION]->(b:Account)
                RETURN a.id AS origen, b.id AS destino, t.amount AS monto, t.isFraud AS fraude
                LIMIT 5
            """).values()
            
            print("\nEjemplos de transacciones:")
            for row in ejemplo:
                print(f"Origen: {row[0]}, Destino: {row[1]}, Monto: {row[2]}, Fraud: {row[3]}")

verificar_datos_neo4j()

Número total de nodos Account: 9073900
Número total de relaciones TRANSACTION: 6362620

Ejemplos de transacciones:
Origen: C1454676819, Destino: C1626852381, Monto: 3740.04, Fraud: 0
Origen: C292773454, Destino: C1626852381, Monto: 55560.91, Fraud: 0
Origen: C1475918760, Destino: C1626852381, Monto: 324505.42, Fraud: 0
Origen: C1489616155, Destino: C1626852381, Monto: 217705.57, Fraud: 0
Origen: C1549778347, Destino: C1626852381, Monto: 167013.95, Fraud: 0


In [13]:

def calcular_grados_nativos_pit(batch_size=40000):
    """
    Calcula in_degree y out_degree históricos para cada transacción
    aplicando Point-in-Time estricto para evitar Data Leakage.
    """
    
    # Consulta maestra que opera en lotes dentro de Neo4j
    query = """
    CALL apoc.periodic.iterate(
      "MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account) RETURN orig, dest, t",
      
      "CALL {
           WITH orig, t
           MATCH (orig)-[out_tx:TRANSACTION]->()
           WHERE out_tx.step < t.step             
           RETURN count(out_tx) AS out_deg
       }
       CALL {
           WITH dest, t
           MATCH ()-[in_tx:TRANSACTION]->(dest)
           WHERE in_tx.step < t.step             
           RETURN count(in_tx) AS in_deg
       }
       SET t.out_degree_hist = out_deg,
           t.in_degree_hist = in_deg",
      
      {batchSize: $batch_size, parallel: false}
    )
    """
    
    print("Iniciando cálculo de características estructurales en Neo4j...")
    print("Modo: Point-in-Time (PIT) | Protección contra Data Leakage: ACTIVA")
    
    start_time = time.time()
    
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:
            # Pasamos el batch_size como parámetro de la consulta
            result = session.run(query, batch_size=batch_size)
            summary = result.single()
            
            print("\n--- Resumen del Procesamiento Nativo ---")
            print(f"Total de transacciones analizadas y enriquecidas: {summary['total']}")
            print(f"Lotes (batches) procesados: {summary['batches']}")
            print(f"Operaciones fallidas: {summary['failedOperations']}")
            
            if summary['errorMessages']:
                print(f"Alertas/Errores reportados: {summary['errorMessages']}")
                
    end_time = time.time()
    print(f"\n--- Características creadas con éxito en {round((end_time - start_time)/60, 2)} minutos ---")


calcular_grados_nativos_pit()

Iniciando cálculo de características estructurales en Neo4j...
Modo: Point-in-Time (PIT) | Protección contra Data Leakage: ACTIVA

--- Resumen del Procesamiento Nativo ---
Total de transacciones analizadas y enriquecidas: 6362620
Lotes (batches) procesados: 160
Operaciones fallidas: 0

--- Características creadas con éxito en 2.45 minutos ---


In [28]:

def calcular_pagerank_temporal_bloques_produccion(uri, auth, tamaño_bloque=24):
    STEP_MAXIMO = 743 
    
    with GraphDatabase.driver(uri, auth=auth) as driver:
        with driver.session() as session:
            
            print("=== INICIANDO PIPELINE MAESTRO DE PAGERANK TEMPORAL ===")
            start_global = time.time()
            
            print("[Paso 0] Inicializando propiedades base en TRANSACTION (Por lotes)...")
            query_inicializacion = """
            CALL apoc.periodic.iterate(
              "MATCH ()-[t:TRANSACTION]->() RETURN t",
              "SET t.orig_pagerank_hist = 1.0, t.dest_pagerank_hist = 1.0",
              {batchSize: 60000, parallel: false}
            )
            """
            session.run(query_inicializacion)
            print("-> Inicialización completada con éxito.")
            
            for inicio_bloque in range(1, STEP_MAXIMO, tamaño_bloque):
                fin_bloque = inicio_bloque + tamaño_bloque - 1
                siguiente_bloque_fin = fin_bloque + tamaño_bloque
                
                print(f"\n[Procesando] Historial <= Step {fin_bloque} | Inyectando en pasos {fin_bloque + 1} al {siguiente_bloque_fin}")
                nombre_proyeccion = f"grafo_bloque_{fin_bloque}"
                
                try:
                    query_proyectar = f"""
                    CALL gds.graph.project.cypher(
                      '{nombre_proyeccion}',
                      'MATCH (a:Account) RETURN id(a) AS id, ["Account"] AS labels',
                      'MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account) WHERE t.step <= {fin_bloque} RETURN id(orig) AS source, id(dest) AS target, "TRANSACTION" AS type'
                    )
                    """
                    session.run(query_proyectar)
                    
                    query_pagerank = f"""
                    CALL gds.pageRank.write(
                      '{nombre_proyeccion}',
                      {{ writeProperty: 'pagerank_temporal' }}
                    )
                    """
                    session.run(query_pagerank)
                    
                    query_congelar = f"""
                    MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account)
                    WHERE t.step > {fin_bloque} AND t.step <= {siguiente_bloque_fin}
                    SET t.orig_pagerank_hist = coalesce(orig.pagerank_temporal, 1.0),
                        t.dest_pagerank_hist = coalesce(dest.pagerank_temporal, 1.0)
                    """
                    session.run(query_congelar)
                    
                    query_limpieza_nodo = f"""
                    MATCH (a:Account) 
                    WHERE a.pagerank_temporal IS NOT NULL
                    CALL apoc.periodic.iterate(
                      "MATCH (n:Account) WHERE n.pagerank_temporal IS NOT NULL RETURN n",
                      "REMOVE n.pagerank_temporal",
                      {{batchSize: 50000, parallel: false}}
                    ) YIELD total RETURN total
                    """
                    session.run(query_limpieza_nodo)
                    
                    print(f"   -> Éxito: Bloque {fin_bloque} procesado e inyectado.")
                    
                except Exception as e:
                    print(f"   -> ERROR EN BLOQUE {fin_bloque}: {e}")
                    
                finally:
                    session.run(f"CALL gds.graph.drop('{nombre_proyeccion}', false)")
            
            print(f"\n=== PIPELINE FINALIZADO CON ÉXITO EN {round((time.time() - start_global)/60, 2)} MINUTOS ===")


calcular_pagerank_temporal_bloques_produccion(URI, AUTH, tamaño_bloque=24)

=== INICIANDO PIPELINE MAESTRO DE PAGERANK TEMPORAL ===
[Paso 0] Inicializando propiedades base en TRANSACTION (Por lotes)...
-> Inicialización completada con éxito.

[Procesando] Historial <= Step 24 | Inyectando en pasos 25 al 48


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. gds.graph.project.cypher is deprecated. It is replaced by gds.graph.project Cypher projection as an aggregation function.', position=<SummaryInputPosition line=2, column=21, offset=21>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 21, 'line': 2, 'column': 21}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n                    CALL gds.graph.project.cypher(\n                      \'grafo_bloque_24\',\n                      \'MATCH (a:Account) RETURN id(a) AS id, ["Account"] AS labels\',\n                      \'MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account) WHERE t.step <= 24 RET

   -> Éxito: Bloque 24 procesado e inyectado.


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('grafo_bloque_24', false)"



[Procesando] Historial <= Step 48 | Inyectando en pasos 49 al 72


...


[Procesando] Historial <= Step 744 | Inyectando en pasos 745 al 768


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. gds.graph.project.cypher is deprecated. It is replaced by gds.graph.project Cypher projection as an aggregation function.', position=<SummaryInputPosition line=2, column=21, offset=21>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 21, 'line': 2, 'column': 21}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n                    CALL gds.graph.project.cypher(\n                      \'grafo_bloque_744\',\n                      \'MATCH (a:Account) RETURN id(a) AS id, ["Account"] AS labels\',\n                      \'MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account) WHERE t.step <= 744 R

   -> Éxito: Bloque 744 procesado e inyectado.


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('grafo_bloque_744', false)"



=== PIPELINE FINALIZADO CON ÉXITO EN 24.35 MINUTOS ===


In [32]:

def calcular_louvain_temporal_bloques_produccion(tamaño_bloque=24):
    STEP_MAXIMO = 743 
    
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:
            
            print("=== INICIANDO PIPELINE MAESTRO DE LOUVAIN TEMPORAL ===")
            start_global = time.time()
            
            print("[Paso 0.1] Asegurando índice estructural TRANSACTION(step)...")
            session.run("CREATE INDEX tx_step_idx IF NOT EXISTS FOR ()-[r:TRANSACTION]-() ON (r.step);")
            
            print("[Paso 0.2] Inicializando propiedades base de Louvain en TRANSACTION (Por lotes)...")
            query_inicializacion = """
            CALL apoc.periodic.iterate(
              "MATCH ()-[t:TRANSACTION]->() RETURN t",
              "SET t.orig_louvain_size_hist = 1.0, t.same_louvain_community_hist = 0",
              {batchSize: 60000, parallel: false}
            )
            """
            session.run(query_inicializacion)
            print("-> Inicialización completada con éxito.")
            
            for inicio_bloque in range(1, STEP_MAXIMO, tamaño_bloque):
                fin_bloque = inicio_bloque + tamaño_bloque - 1
                siguiente_bloque_fin = fin_bloque + tamaño_bloque
                
                print(f"\n[Procesando] Historial Louvain <= Step {fin_bloque} | Inyectando en pasos {fin_bloque + 1} al {siguiente_bloque_fin}")
                nombre_proyeccion = f"grafo_louvain_bloque_{fin_bloque}"
                
                try:
                    query_proyectar = f"""
                    CALL gds.graph.project.cypher(
                      '{nombre_proyeccion}',
                      'MATCH (a:Account) RETURN id(a) AS id, ["Account"] AS labels',
                      'MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account) WHERE t.step <= {fin_bloque} RETURN id(orig) AS source, id(dest) AS target, "TRANSACTION" AS type'
                    )
                    """
                    session.run(query_proyectar)
                    
                    query_louvain = f"""
                    CALL gds.louvain.write(
                      '{nombre_proyeccion}',
                      {{ 
                        writeProperty: 'louvain_id_temp',
                        includeIntermediateCommunities: false,
                        concurrency: 4
                      }}
                    )
                    """
                    session.run(query_louvain)
                    
                    query_congelar_lotes = f"""
                    CALL apoc.periodic.iterate(
                      "MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account) WHERE t.step > {fin_bloque} AND t.step <= {siguiente_bloque_fin} RETURN t, orig, dest",
                      "SET t.orig_louvain_size_hist = 1.0,
                           t.same_louvain_community_hist = CASE 
                               WHEN orig.louvain_id_temp IS NOT NULL 
                                    AND dest.louvain_id_temp IS NOT NULL 
                                    AND orig.louvain_id_temp = dest.louvain_id_temp THEN 1 
                               ELSE 0 
                           END",
                      {{batchSize: 40000, parallel: false}}
                    )
                    """
                    session.run(query_congelar_lotes)
                    
                    query_limpieza_nodo = """
                    CALL apoc.periodic.iterate(
                      "MATCH (n:Account) WHERE n.louvain_id_temp IS NOT NULL RETURN n",
                      "REMOVE n.louvain_id_temp",
                      {batchSize: 60000, parallel: false}
                    ) YIELD total RETURN total
                    """
                    session.run(query_limpieza_nodo)
                    
                    print(f"   -> Éxito: Bloque {fin_bloque} procesado de forma segura.")
                    
                except Exception as e:
                    print(f"   -> ⚠️ ERROR EN BLOQUE {fin_bloque}: {e}")
                    
                finally:
                    session.run(f"CALL gds.graph.drop('{nombre_proyeccion}', false)")
            
            print(f"\n=== PIPELINE LOUVAIN FINALIZADO CON ÉXITO EN {round((time.time() - start_global)/60, 2)} MINUTOS ===")

calcular_louvain_temporal_bloques_produccion(tamaño_bloque=24)

=== INICIANDO PIPELINE MAESTRO DE LOUVAIN TEMPORAL ===
[Paso 0.1] Asegurando índice estructural TRANSACTION(step)...
[Paso 0.2] Inicializando propiedades base de Louvain en TRANSACTION (Por lotes)...
-> Inicialización completada con éxito.

[Procesando] Historial Louvain <= Step 24 | Inyectando en pasos 25 al 48


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. gds.graph.project.cypher is deprecated. It is replaced by gds.graph.project Cypher projection as an aggregation function.', position=<SummaryInputPosition line=2, column=21, offset=21>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 21, 'line': 2, 'column': 21}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n                    CALL gds.graph.project.cypher(\n                      \'grafo_louvain_bloque_24\',\n                      \'MATCH (a:Account) RETURN id(a) AS id, ["Account"] AS labels\',\n                      \'MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account) WHERE t.step <

   -> Éxito: Bloque 24 procesado de forma segura.


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('grafo_louvain_bloque_24', false)"



[Procesando] Historial Louvain <= Step 48 | Inyectando en pasos 49 al 72


...


[Procesando] Historial Louvain <= Step 744 | Inyectando en pasos 745 al 768


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. gds.graph.project.cypher is deprecated. It is replaced by gds.graph.project Cypher projection as an aggregation function.', position=<SummaryInputPosition line=2, column=21, offset=21>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 21, 'line': 2, 'column': 21}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n                    CALL gds.graph.project.cypher(\n                      \'grafo_louvain_bloque_744\',\n                      \'MATCH (a:Account) RETURN id(a) AS id, ["Account"] AS labels\',\n                      \'MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account) WHERE t.step 

   -> Éxito: Bloque 744 procesado de forma segura.


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('grafo_louvain_bloque_744', false)"



=== PIPELINE LOUVAIN FINALIZADO CON ÉXITO EN 21.25 MINUTOS ===


In [35]:

def exportacion_absoluta_a_csv(archivo_salida="master_dataset.csv.gz"):
    print(f"=== INICIANDO EXPORTACIÓN ABSOLUTA Y FUTURA A {archivo_salida} ===")
    start_time = time.time()
    
    # Query maximizada: Extrae TODO lo estructural, lo original de PaySim y lo calculado por GDS PIT
    query = """
    MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account)
    RETURN 
        t.step AS step,
        orig.id AS nameOrig,
        dest.id AS nameDest,
        t.type AS type,
        t.amount AS amount,
        
        t.oldBalanceOrig AS oldbalanceOrg,
        t.newBalanceOrig AS newbalanceOrig,
        t.oldBalanceDest AS oldbalanceDest,
        t.newBalanceDest AS newbalanceDest,
        
        t.isFraud AS isFraud,
        t.isFlaggedFraud AS isFlaggedFraud,
        
        t.in_degree_hist AS in_degree_hist,
        t.out_degree_hist AS out_degree_hist,
        
        t.orig_pagerank_hist AS orig_pagerank_hist,
        t.dest_pagerank_hist AS dest_pagerank_hist,
        t.same_louvain_community_hist AS same_louvain_community_hist
    """
    
    # Mapeo idéntico de cabeceras para el archivo CSV
    columnas = [
        "step", "nameOrig", "nameDest", "type", "amount",
        "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest",
        "isFraud", "isFlaggedFraud",
        "in_degree_hist", "out_degree_hist", 
        "orig_pagerank_hist", "dest_pagerank_hist", "same_louvain_community_hist"
    ]
    
    contador = 0
    
    # Escribimos comprimiendo sobre la marcha para asegurar espacio en disco
    with gzip.open(archivo_salida, mode='wt', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(columnas) # Cabecera
        
        with GraphDatabase.driver(URI, auth=AUTH) as driver:
            with driver.session() as session:
                result = session.run(query)
                
                for record in result:
                    writer.writerow([record[col] for col in columnas])
                    contador += 1
                    
                    if contador % 500000 == 0:
                        print(f"-> {contador} registros volcados a disco...")
                        
    print(f"\n=== PROCESO COMPLETADO EN {round((time.time() - start_time)/60, 2)} MINUTOS ===")
    print(f"Registros totales salvados con éxito: {contador}")
    print(f"Tu CSV Maestro está listo y blindado contra cambios futuros en: {archivo_salida}")


exportacion_absoluta_a_csv("../data/processed/master_dataset")

=== INICIANDO EXPORTACIÓN ABSOLUTA Y FUTURA A ../data/processed/master_dataset ===
-> 500000 registros volcados a disco...
-> 1000000 registros volcados a disco...
-> 1500000 registros volcados a disco...
-> 2000000 registros volcados a disco...
-> 2500000 registros volcados a disco...
-> 3000000 registros volcados a disco...
-> 3500000 registros volcados a disco...
-> 4000000 registros volcados a disco...
-> 4500000 registros volcados a disco...
-> 5000000 registros volcados a disco...
-> 5500000 registros volcados a disco...
-> 6000000 registros volcados a disco...


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `newBalanceOrig` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=11, column=11, offset=266>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 266, 'line': 11, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account)\n    RETURN \n        t.step AS step,\n        orig.id AS nameOrig,\n        dest.id AS nameDest,\n        t.type AS type,\n        t.amount AS amount,\n        \n        t.oldBalanceOrig AS oldbalanceOrg,\n        t.newBalanceOrig AS newbalance


=== PROCESO COMPLETADO EN 4.81 MINUTOS ===
Registros totales salvados con éxito: 6362620
Tu CSV Maestro está listo y blindado contra cambios futuros en: ../data/processed/master_dataset


In [3]:
def exportacion_absoluta_corregida_a_csv(archivo_salida="../data/processed/master_dataset_v2.csv.gz"):
    print(f"=== INICIANDO EXPORTACIÓN ABSOLUTA Y FUTURA A {archivo_salida} ===")
    start_time = time.time()
    
    query = """
    MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account)
    RETURN 
        t.step AS step,
        orig.id AS nameOrig,
        dest.id AS nameDest,
        t.type AS type,
        t.amount AS amount,
        t.oldbalanceOrg AS oldbalanceOrg,
        t.newbalanceOrig AS newbalanceOrig,
        t.oldbalanceDest AS oldbalanceDest,
        t.newbalanceDest AS newbalanceDest,
        t.isFraud AS isFraud,
        t.isFlaggedFraud AS isFlaggedFraud,
        t.in_degree_hist AS in_degree_hist,
        t.out_degree_hist AS out_degree_hist,
        t.orig_pagerank_hist AS orig_pagerank_hist,
        t.dest_pagerank_hist AS dest_pagerank_hist,
        t.same_louvain_community_hist AS same_louvain_community_hist
    """
    
    columnas = [
        "step", "nameOrig", "nameDest", "type", "amount",
        "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest",
        "isFraud", "isFlaggedFraud",
        "in_degree_hist", "out_degree_hist",
        "orig_pagerank_hist", "dest_pagerank_hist", "same_louvain_community_hist"
    ]
    
    contador = 0
    
    with gzip.open(archivo_salida, mode='wt', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(columnas)
        
        with GraphDatabase.driver(URI, auth=AUTH) as driver:
            with driver.session() as session:
                result = session.run(query)
                
                for record in result:
                    writer.writerow([record[col] for col in columnas])
                    contador += 1
                    
                    if contador % 500000 == 0:
                        print(f"-> {contador} registros volcados a disco...")
                        
    print(f"\n=== PROCESO COMPLETADO EN {round((time.time() - start_time)/60, 2)} MINUTOS ===")
    print(f"Registros totales salvados con éxito: {contador}")
    print(f"Tu CSV Maestro está listo y blindado en: {archivo_salida}")

exportacion_absoluta_corregida_a_csv("../data/processed/master_dataset_v2.csv.gz")

=== INICIANDO EXPORTACIÓN ABSOLUTA Y FUTURA A ../data/processed/master_dataset_v2.csv.gz ===
-> 500000 registros volcados a disco...
-> 1000000 registros volcados a disco...
-> 1500000 registros volcados a disco...
-> 2000000 registros volcados a disco...
-> 2500000 registros volcados a disco...
-> 3000000 registros volcados a disco...
-> 3500000 registros volcados a disco...
-> 4000000 registros volcados a disco...
-> 4500000 registros volcados a disco...
-> 5000000 registros volcados a disco...
-> 5500000 registros volcados a disco...
-> 6000000 registros volcados a disco...


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `isFlaggedFraud` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=14, column=11, offset=418>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 418, 'line': 14, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (orig:Account)-[t:TRANSACTION]->(dest:Account)\n    RETURN \n        t.step AS step,\n        orig.id AS nameOrig,\n        dest.id AS nameDest,\n        t.type AS type,\n        t.amount AS amount,\n        t.oldbalanceOrg AS oldbalanceOrg,\n        t.newbalanceOrig AS newbalanceOrig,\n    


=== PROCESO COMPLETADO EN 5.08 MINUTOS ===
Registros totales salvados con éxito: 6362620
Tu CSV Maestro está listo y blindado en: ../data/processed/master_dataset_v2.csv.gz
